In [6]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
import re

START_TIME = datetime.now()

def log(msg):
    now = datetime.now()
    elapsed = (now - START_TIME).total_seconds()
    print(f"[{now:%Y-%m-%d %H:%M:%S} | +{elapsed:.1f}s] {msg}")

def read_csv_auto(path):
    encodings = ["utf-8-sig", "utf-8", "cp932", "shift_jis"]
    last_error = None

    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception as e:
            last_error = e

    raise last_error

def extract_datetimes_from_summary(df):
    # 候補列
    candidates = [
        "cp_datetime",
        "cp_datetimes",
        "cp_date",
        "cp_dates",
        "cp_timestamp",
        "cp_timestamps",
        "change_point_datetime",
        "change_point_datetimes",
        "change_point_date",
        "change_point_dates",
        "change_points",
        "change_point_times",
        "cp_times",
    ]

    existing = [c for c in candidates if c in df.columns]

    if not existing:
        return pd.Series(dtype="datetime64[ns]")

    values = []

    for col in existing:
        for v in df[col].dropna().astype(str):
            if not v.strip():
                continue

            # 日時らしき文字列を抽出
            found = re.findall(
                r"\d{4}-\d{2}-\d{2}(?:[ T]\d{2}:\d{2}:\d{2})?",
                v
            )

            values.extend(found)

    if not values:
        return pd.Series(dtype="datetime64[ns]")

    return pd.to_datetime(values, errors="coerce").dropna()

# =========================================
# 1. ファイル検索
# =========================================
log("ファイル検索開始")

current_dir = Path(".")

files = sorted(
    current_dir.glob("*change_point_summary*.csv"),
    key=lambda f: f.stat().st_mtime
)

if not files:
    raise FileNotFoundError("change_point_summary を含むCSVが見つかりません")

out_dir = current_dir / "daily_change_point_plots"
out_dir.mkdir(exist_ok=True)

log(f"対象ファイル数: {len(files)}")
log(f"出力先: {out_dir.resolve()}")

success_count = 0
skip_count = 0
error_count = 0

# =========================================
# 2. ファイルごとに処理
# =========================================
for i, file_path in enumerate(files, start=1):
    log(f"処理中 ({i}/{len(files)}): {file_path.name}")

    try:
        if file_path.stat().st_size == 0:
            log(f"スキップ 空ファイル: {file_path.name}")
            skip_count += 1
            continue

        df = read_csv_auto(file_path)

        if df.empty:
            log(f"スキップ データなし: {file_path.name}")
            skip_count += 1
            continue

        datetimes = extract_datetimes_from_summary(df)

        if datetimes.empty:
            log(f"スキップ 日時列なし: {file_path.name}")
            log(f"列一覧: {list(df.columns)}")
            skip_count += 1
            continue

        tmp = pd.DataFrame({"datetime": datetimes})
        tmp["date"] = tmp["datetime"].dt.date

        daily_counts = tmp.groupby("date").size().sort_index()

        title_name = file_path.stem
        if len(title_name) > 60:
            title_name = title_name[:60] + "..."

        plt.figure(figsize=(12, 5))
        daily_counts.plot(kind="bar")

        plt.xlabel("date")
        plt.ylabel("change point count")
        plt.title(f"Daily Change Points\n{title_name}")
        plt.xticks(rotation=45)
        plt.tight_layout()

        out_png = out_dir / f"{file_path.stem}_daily_counts.png"
        plt.savefig(out_png, dpi=150)
        plt.close()

        log(f"保存完了: {out_png.name}")
        success_count += 1

    except Exception as e:
        log(f"エラー: {file_path.name} / {e}")
        error_count += 1

log("====================================")
log("処理結果")
log(f"成功: {success_count}")
log(f"スキップ: {skip_count}")
log(f"エラー: {error_count}")
log("完了")

[2026-05-10 13:16:44 | +0.0s] ファイル検索開始
[2026-05-10 13:16:44 | +0.0s] 対象ファイル数: 26
[2026-05-10 13:16:44 | +0.0s] 出力先: D:\musashino-university\finance\daily_change_point_plots
[2026-05-10 13:16:44 | +0.1s] 処理中 (1/26): change_point_summary_pelt_l1_20260507_104255.csv
[2026-05-10 13:16:45 | +0.6s] スキップ 日時列なし: change_point_summary_pelt_l1_20260507_104255.csv
[2026-05-10 13:16:45 | +0.7s] 列一覧: ['method', 'model', 'file', 'symbol', 'rows', 'return_rows', 'rmse', 'n_change_points', 'status', 'error', 'file_time']
[2026-05-10 13:16:45 | +0.7s] 処理中 (2/26): change_point_summary_pelt_l2_20260507_114509.csv
[2026-05-10 13:16:45 | +0.8s] スキップ 日時列なし: change_point_summary_pelt_l2_20260507_114509.csv
[2026-05-10 13:16:45 | +0.8s] 列一覧: ['method', 'model', 'file', 'symbol', 'rows', 'return_rows', 'rmse', 'n_change_points', 'status', 'error', 'file_time']
[2026-05-10 13:16:45 | +0.8s] 処理中 (3/26): change_point_summary_pelt_rbf_20260507_145230.csv
[2026-05-10 13:16:45 | +1.0s] スキップ 日時列なし: change_point_summar